In [2]:
import cv2
import numpy as np

# 设置图像大小
width = 255
height = 255

# 创建一个空白图像 (白色背景)
img = np.ones((height, width, 3), np.uint8) * 255  # 3通道，白色

# 设置点的大小和颜色
dot_size = 30
dot_color = (0, 0, 0)  # 黑色

# 计算点之间的间距
x_spacing = width // 5
y_spacing = height // 5

# 循环绘制点
for i in range(5):
    for j in range(5):
        # 计算点的中心坐标
        x = x_spacing * j + x_spacing // 2
        y = y_spacing * i + y_spacing // 2

        # 绘制圆形点
        cv2.circle(img, (x, y), 5, dot_color, -1)  # -1 填充圆形

# 显示图像
cv2.imshow("5x5 Dot Grid", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

cv2.imwrite("5x5_dot_grid.png", img)


ModuleNotFoundError: No module named 'cv2'

In [27]:
import cv2
import numpy as np
import random

# 设置图像大小
width = 255
height = 255

# 创建一个空白图像 (白色背景)
img = np.ones((height, width, 3), np.uint8) * 255  # 3通道，白色

# 设置点的大小和颜色
dot_size = 10
dot_color = (0, 0, 0)  # 黑色
line_color = (0, 0, 0)  # 黑色线条

# 计算点之间的间距
x_spacing = width // 5
y_spacing = height // 5

# 创建一个点坐标的列表
points = []
for i in range(5):
    for j in range(5):
        x = x_spacing * j + x_spacing // 2
        y = y_spacing * i + y_spacing // 2
        points.append((x, y))

for i, (x, y) in enumerate(points):
    cv2.circle(img, (x, y), dot_size // 2, dot_color, -1)

# 随机选择起点
start_index = random.randint(0, 24)
start_point = points[start_index]
print(f"起点是点 {start_index + 1}")

# 在起始点周围画一个圆圈
cv2.circle(img, start_point, dot_size, (0, 0, 0), 2)  # 红色圆圈

# 生成随机线段
num_segments = random.randint(4, 6)
current_point = start_point
current_index = start_index
visited_indices = {start_index}  # 记录已经访问过的点的索引
lines = []  # 记录已绘制的线段
blocked_points = set()  # 记录被阻塞的点

def lines_intersect(p1, p2, p3, p4):
    """检查线段 (p1, p2) 与 (p3, p4) 是否相交"""
    def ccw(a, b, c):
        return (c[1] - a[1]) * (b[0] - a[0]) > (b[1] - a[1]) * (c[0] - a[0])

    return ccw(p1, p3, p4) != ccw(p2, p3, p4) and ccw(p1, p2, p3) != ccw(p1, p2, p4)

def block_points(start_idx, end_idx):
    """根据起点和终点阻塞中间的点"""
    start_row, start_col = divmod(start_idx, 5)
    end_row, end_col = divmod(end_idx, 5)

    if start_row == end_row:  # 同一行
        if start_col < end_col:
            for col in range(start_col + 1, end_col):
                blocked_points.add(start_row * 5 + col)
        else:
            for col in range(end_col + 1, start_col):
                blocked_points.add(start_row * 5 + col)
    elif start_col == end_col:  # 同一列
        if start_row < end_row:
            for row in range(start_row + 1, end_row):
                blocked_points.add(row * 5 + start_col)
        else:
            for row in range(end_row + 1, start_row):
                blocked_points.add(row * 5 + start_col)

for _ in range(num_segments):
    # 随机选择下一个点 (不能是当前点, 且不能是已经访问过的点)
    possible_next_indices = [i for i in range(25) if i not in visited_indices and i not in blocked_points]
    if not possible_next_indices:  # 如果没有其他点可用，结束循环
        print("没有未访问的点，结束连接。")
        break

    next_index = None
    random.shuffle(possible_next_indices)  # 打乱可选点的顺序
    for index in possible_next_indices:
        next_point = points[index]
        # 检查当前段与已绘制段是否有交叉
        if all(not lines_intersect(current_point, next_point, lines[i][0], lines[i][1]) for i in range(len(lines))):
            next_index = index
            break

    if next_index is not None:
        block_points(current_index, next_index)  # 阻塞中间的点
        next_point = points[next_index]
        # 绘制线段
        cv2.line(img, current_point, next_point, line_color, 2)
        lines.append((current_point, next_point))  # 记录已绘制的线段

        # 更新当前点和索引
        current_point = next_point
        current_index = next_index  # 更新当前索引
        visited_indices.add(next_index)  # 将新点添加到已访问集合
        print(f"连接到点 {next_index + 1}")
    else:
        print("没有有效的下一个点，结束连接。")
        break

# 显示图像
cv2.imshow("5x5 Dot Grid with Non-Intersecting Lines", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

# 可选：保存图像
cv2.imwrite("5x5_dot_grid_non_intersecting_lines.png", img)

起点是点 2
连接到点 11
连接到点 25
连接到点 10
连接到点 12


True

In [41]:
print(f"{next_index//5+1,next_index%5+1}")

(3, 1)


In [28]:
import cv2
# 保存生成的图像
output_filename = "generated_image.png"
cv2.imwrite(output_filename, img)

# 加载 1.png 图片
try:
    image1 = cv2.imread("1.png")
    if image1 is None:
        raise FileNotFoundError("1.png not found or could not be opened.")
except FileNotFoundError as e:
    print(f"Error: {e}")
    exit()

# 加载生成的图像
image2 = cv2.imread(output_filename)

# 确保两张图片都成功加载
if image1 is None or image2 is None:
    print("Error: Could not load one or both images.")
    exit()

# 调整图片大小，使高度一致 (可选，如果需要)
height1, width1, _ = image1.shape
height2, width2, _ = image2.shape

if height1 != height2:
  print("图像高度不一致，请确保两张图片高度一致。")
  exit()

# 定义间隔大小
gap_size = 80  # 可以根据需要调整

# 创建一个空白的间隔图像
gap = np.ones((height1, gap_size, 3), np.uint8) * 255  # 白色间隔

# 水平拼接图像和间隔
combined_image = np.concatenate((image2, gap, image1), axis=1)

# 显示合并后的图像
cv2.imshow("Combined Image", combined_image)
cv2.waitKey(0)
cv2.destroyAllWindows()

# 保存合并后的图像
cv2.imwrite("combined_image.png", combined_image)

True